# Synthetic Training Data Generator

Generate high-quality synthetic datasets for fine-tuning or evaluation using LLMs and HuggingFace tools.

This project takes a **topic** and a **data type** as input, then generates **structured synthetic data** such as:
- Q&A pairs
- sentiment analysis samples

The generated data is further analyzed using **HuggingFace tokenizers** and can be exported in **JSON format** for downstream use.

### Week 3 Community Contribution
**By Saimadhu Polamuri**

### Skills Demonstrated
- HuggingFace Transformers
- Tokenizers
- Open-source model integration
- Synthetic data generation

## Import Required Libraries

In this cell, we import the libraries needed for the synthetic training data generator.

- `os` is used to access environment variables such as API keys.
- `json` is used to structure and export generated synthetic data in JSON format.
- `re` is used for text cleaning and pattern matching when processing model outputs.
- `OpenAI` is used to connect to language models for generating synthetic training samples.
- `AutoTokenizer` from HuggingFace Transformers is used to tokenize and analyze the generated text data.

In [ ]:
# Standard library imports
import os
import json
import re

# Third-party imports
from openai import OpenAI
from transformers import AutoTokenizer

## Configure the LLM Client and Model

In this cell, we initialize the OpenAI-compatible client that will be used to generate synthetic training data.

- `base_url` is loaded from the environment if available, and defaults to the OpenRouter API endpoint.
- `api_key` is read from the `OPENAI_API_KEY` environment variable.
- `MODEL` specifies the language model used for generation.

This setup makes the notebook flexible, since it can work with different OpenAI-compatible providers by changing environment variables instead of modifying the code.

In [ ]:
# Initialize the OpenAI-compatible client
client = OpenAI(
    base_url=os.environ.get("OPENAI_BASE_URL", "https://openrouter.ai/api/v1"),
    api_key=os.environ.get("OPENAI_API_KEY")
)

# Define the model to use for synthetic data generation
MODEL = "gpt-4.1-nano"

## Load the HuggingFace Tokenizer

In this cell, we load the `gpt2` tokenizer from HuggingFace Transformers.

The tokenizer is used to analyze the synthetic text generated by the language model. This helps us understand how the text is split into tokens, which is useful for tasks such as:
- estimating input size for model training or evaluation
- checking token efficiency
- preparing data for downstream fine-tuning workflows

The cell also prints basic tokenizer details such as the tokenizer name and vocabulary size.

In [ ]:
# Load a HuggingFace tokenizer to analyze generated text data
tokenizer = AutoTokenizer.from_pretrained("gpt2")

# Display basic tokenizer information
print(f"Loaded tokenizer: {tokenizer.name_or_path}")
print(f"Vocab size: {tokenizer.vocab_size:,}")

## Generate Synthetic Q&A Pairs

In this cell, we define a function that generates synthetic question-answer pairs for a given topic using an LLM.

The function:
- accepts a topic, number of pairs, and difficulty level
- constructs a structured prompt requesting JSON-only output
- calls the selected language model
- cleans the response in case the model wraps the JSON in markdown code fences
- parses the JSON output into Python data structures

This function is useful for creating synthetic training or evaluation data that can later be analyzed, filtered, or exported.

In [ ]:
def generate_qa_pairs(topic, num_pairs=5, difficulty="mixed"):
    """
    Generate synthetic question-answer pairs for a given topic.

    The function sends a structured prompt to the language model and expects
    the response in strict JSON array format. It then cleans the response
    and parses it into Python objects.

    Args:
        topic (str): Topic for which Q&A pairs should be generated.
        num_pairs (int, optional): Number of question-answer pairs to generate.
                                   Defaults to 5.
        difficulty (str, optional): Difficulty level of the generated pairs.
                                    Defaults to "mixed".

    Returns:
        list: Parsed list of generated Q&A dictionaries.
    """
    prompt = f"""Generate exactly {num_pairs} question-answer pairs about: {topic}
Difficulty level: {difficulty}

Respond ONLY with a valid JSON array, no markdown, no extra text:
[
  {{"question": "...", "answer": "...", "difficulty": "easy|medium|hard"}}
]

Make questions diverse - mix factual, conceptual, and applied questions."""

    # Send the prompt to the model
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}]
    )

    # Extract the raw text response
    raw = response.choices[0].message.content.strip()

    # Remove markdown code fences if the model returns JSON inside them
    raw = re.sub(r"^```json\s*", "", raw)
    raw = re.sub(r"\s*```$", "", raw)

    # Parse and return the JSON response
    return json.loads(raw)

## Generate and Display Sample Q&A Data

In this cell, we use the `generate_qa_pairs()` function to create a sample synthetic dataset on the topic **Python decorators**.

The generated Q&A pairs are then printed in a readable format, showing:
- the question
- the answer
- the assigned difficulty level

This step helps verify that the data generation function is working correctly and that the output is diverse, structured, and useful for downstream tasks.

In [ ]:
# Generate synthetic Q&A pairs for the selected topic
qa_data = generate_qa_pairs("Python decorators", num_pairs=5)

# Display each generated question-answer pair in a readable format
for i, pair in enumerate(qa_data, 1):
    print(f"Q{i}: {pair['question']}")
    print(f"A{i}: {pair['answer']}")
    print(f"    Difficulty: {pair['difficulty']}\n")

## Generate Synthetic Sentiment Analysis Data

In this cell, we define a function that generates synthetic sentiment-labeled text samples for a chosen domain using an LLM.

The function:
- accepts a target domain and the number of samples to generate
- requests realistic text examples with sentiment labels and confidence scores
- enforces JSON-only output
- cleans markdown formatting if the model wraps the JSON in code fences
- parses the JSON output into Python objects

This function is useful for building synthetic datasets for sentiment classification experiments, model evaluation, or fine-tuning tasks.

In [ ]:
def generate_sentiment_data(domain, num_samples=10):
    """
    Generate synthetic sentiment-labeled text samples for a given domain.

    The function prompts the language model to create realistic sentiment
    analysis examples and expects the output in strict JSON array format.
    It then removes any markdown formatting and parses the JSON response.

    Args:
        domain (str): The domain for which sentiment samples should be generated.
                      Example: product reviews, customer support, or movies.
        num_samples (int, optional): Number of sentiment samples to generate.
                                     Defaults to 10.

    Returns:
        list: Parsed list of sentiment-labeled text dictionaries.
    """
    prompt = f"""Generate exactly {num_samples} realistic text samples for sentiment analysis in the domain: {domain}

Respond ONLY with a valid JSON array, no markdown:
[
  {{"text": "...", "sentiment": "positive|negative|neutral", "confidence": 0.0-1.0}}
]

Balance the sentiments roughly equally. Make texts realistic and varied in length."""

    # Send the prompt to the model
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}]
    )

    # Extract the raw text response
    raw = response.choices[0].message.content.strip()

    # Remove markdown code fences if present
    raw = re.sub(r"^```json\s*", "", raw)
    raw = re.sub(r"\s*```$", "", raw)

    # Parse and return the JSON response
    return json.loads(raw)

## Generate and Display Sample Sentiment Data

In this cell, we use the `generate_sentiment_data()` function to create a synthetic sentiment analysis dataset for the domain **restaurant reviews**.

The generated samples are printed in a readable format, showing:
- the sentiment label
- the confidence score
- the generated review text

This step helps verify that the sentiment data generator is producing balanced, realistic, and properly structured output.

In [ ]:
# Generate synthetic sentiment analysis samples for the selected domain
sentiment_data = generate_sentiment_data("restaurant reviews", num_samples=6)

# Display each generated sample with its sentiment label and confidence score
for i, sample in enumerate(sentiment_data, 1):
    print(f"[{sample['sentiment'].upper():>8}] ({sample['confidence']:.1f}) {sample['text']}\n")

## Analyze Dataset Token Statistics

In this cell, we define a function to analyze the token distribution of a generated dataset using a HuggingFace tokenizer.

The function:
- supports both regular text datasets and Q&A datasets
- tokenizes each sample using the loaded tokenizer
- calculates total, average, minimum, and maximum token counts
- displays a sample tokenization of the first dataset entry
- returns the token statistics as a dictionary

This analysis is useful for understanding dataset size, token efficiency, and how well the generated data fits within transformer model input limits.

In [ ]:
def analyze_dataset_tokens(dataset, text_field="text"):
    """
    Analyze token statistics of a dataset using the HuggingFace tokenizer.

    This function supports both standard text datasets and Q&A datasets.
    For Q&A data, it combines the question and answer into a single text
    sequence before tokenization.

    Args:
        dataset (list): List of dataset records to analyze.
        text_field (str, optional): Field name containing the text to tokenize.
                                    Use "qa" for question-answer datasets.
                                    Defaults to "text".

    Returns:
        dict: Dictionary containing total token count, average tokens per sample,
              and token counts for each record.
    """
    # Prepare the text data for token analysis
    if text_field == "qa":
        texts = [f"{item['question']} {item['answer']}" for item in dataset]
    else:
        texts = [item[text_field] for item in dataset]

    # Compute token counts for each text sample
    token_counts = []
    for text in texts:
        tokens = tokenizer.encode(text)
        token_counts.append(len(tokens))

    total_tokens = sum(token_counts)
    avg_tokens = total_tokens / len(token_counts)

    # Print dataset-level token statistics
    print(f"Dataset size: {len(texts)} samples")
    print(f"Total tokens: {total_tokens:,}")
    print(f"Avg tokens/sample: {avg_tokens:.1f}")
    print(f"Min tokens: {min(token_counts)}")
    print(f"Max tokens: {max(token_counts)}")

    # Show sample tokenization for the first dataset entry
    first_tokens = tokenizer.encode(texts[0])
    decoded = [tokenizer.decode([t]) for t in first_tokens[:15]]
    print(f"\nSample tokenization (first entry):")
    print(f"  First 15 tokens: {decoded}")

    return {"total": total_tokens, "avg": avg_tokens, "counts": token_counts}

## Run Token Analysis on Generated Datasets

In this cell, we apply the token analysis function to both generated datasets:

- the synthetic Q&A dataset
- the synthetic sentiment dataset

This helps compare the token distribution across different types of generated data and provides useful insights such as total token usage, average sample length, and tokenization patterns.

In [ ]:
# Analyze token statistics for the generated Q&A dataset
print("=== Q&A Dataset Token Analysis ===")
qa_stats = analyze_dataset_tokens(qa_data, text_field="qa")

# Analyze token statistics for the generated sentiment dataset
print("\n=== Sentiment Dataset Token Analysis ===")
sent_stats = analyze_dataset_tokens(sentiment_data, text_field="text")

## Convert Q&A Data to Fine-Tuning Format

In this cell, we define a function that converts the generated Q&A pairs into a chat-style format suitable for fine-tuning.

The function transforms each Q&A pair into a structure with:
- a `user` message containing the question
- an `assistant` message containing the answer

After conversion, the cell prints:
- the first formatted sample
- the total number of fine-tuning records created

This step prepares the synthetic Q&A dataset for downstream model training workflows.

In [ ]:
def export_as_finetune_jsonl(qa_pairs):
    """
    Convert synthetic Q&A pairs into OpenAI fine-tuning style JSONL records.

    Each Q&A pair is transformed into a chat-style message structure
    containing:
    - a user message with the question
    - an assistant message with the answer

    Args:
        qa_pairs (list): List of question-answer dictionaries.

    Returns:
        list: List of fine-tuning formatted records.
    """
    finetune_data = []

    for pair in qa_pairs:
        entry = {
            "messages": [
                {"role": "user", "content": pair["question"]},
                {"role": "assistant", "content": pair["answer"]}
            ]
        }
        finetune_data.append(entry)

    return finetune_data


# Convert the generated Q&A dataset into fine-tuning format
finetune_ready = export_as_finetune_jsonl(qa_data)

# Display the first formatted sample
print(json.dumps(finetune_ready[0], indent=2))

# Display the total number of fine-tuning samples
print(f"\nTotal fine-tuning samples: {len(finetune_ready)}")

## Generate a Bulk Multi-Topic Dataset

In this cell, we define a function to generate a larger synthetic Q&A dataset across multiple topics.

The function:
- loops through a list of topics
- generates a fixed number of Q&A pairs for each topic
- adds the topic name to each generated record
- combines all records into a single dataset

After generating the bulk dataset, the cell performs token analysis on the combined data to understand its overall size and token distribution.

This step demonstrates how the synthetic data generator can scale beyond a single topic and produce a more diverse dataset for training or evaluation.

In [ ]:
def generate_bulk_dataset(topics, pairs_per_topic=3):
    """
    Generate a larger synthetic Q&A dataset across multiple topics.

    For each topic, this function generates a fixed number of Q&A pairs,
    adds the topic name to each record, and combines everything into
    one dataset.

    Args:
        topics (list): List of topics for which Q&A pairs should be generated.
        pairs_per_topic (int, optional): Number of Q&A pairs to generate
                                         for each topic. Defaults to 3.

    Returns:
        list: Combined list of generated Q&A pairs across all topics.
    """
    all_pairs = []

    for topic in topics:
        print(f"Generating data for: {topic}")
        pairs = generate_qa_pairs(topic, num_pairs=pairs_per_topic)

        # Attach the topic name to each generated pair
        for pair in pairs:
            pair["topic"] = topic

        all_pairs.extend(pairs)

    print(f"\nTotal samples generated: {len(all_pairs)}")
    return all_pairs


# Define multiple topics for bulk synthetic data generation
topics = ["Python list comprehensions", "REST API design", "Docker containers"]

# Generate Q&A pairs across all topics
bulk_data = generate_bulk_dataset(topics, pairs_per_topic=3)

# Analyze token statistics for the combined dataset
print("\n=== Bulk Dataset Token Analysis ===")
bulk_stats = analyze_dataset_tokens(bulk_data, text_field="qa")